In [ ]:
import torch
import torchvision.models as models

## Guardar y Cargar Pesos del Modelo

Los modelos de PyTorch almacenan los parámetros aprendidos en un diccionario de estado interno, llamado `state_dict`. Estos pueden persistirse mediante el método `torch.save`.

### ¿Qué es el state_dict?

El `state_dict` es un diccionario de Python que mapea cada capa del modelo a sus parámetros (pesos y sesgos). Por ejemplo:

```python
{
    'conv1.weight': tensor([...]),
    'conv1.bias': tensor([...]),
    'fc1.weight': tensor([...]),
    'fc1.bias': tensor([...])
}
```

### Ventajas de guardar solo state_dict:

- **Flexibilidad**: Separas la arquitectura del modelo de sus pesos
- **Tamaño**: Archivos más pequeños (solo parámetros, no código)
- **Portabilidad**: Menos problemas al actualizar versiones de PyTorch
- **Seguridad**: Evita ejecutar código arbitrario al cargar

In [ ]:
# Cargar un modelo preentrenado VGG16 con pesos de ImageNet
model = models.vgg16(weights='IMAGENET1K_V1')

# Guardar solo los pesos del modelo
torch.save(model.state_dict(), 'model_weights.pth')

## Cargar Pesos del Modelo

Para cargar los pesos del modelo, primero necesitas crear una instancia de la misma clase del modelo, y luego cargar los parámetros usando el método `load_state_dict()`.

En el código siguiente, configuramos `weights_only=True` para limitar las funciones ejecutadas durante el desencapsulado (*unpickling*) solo a aquellas necesarias para cargar pesos. Usar `weights_only=True` se considera una mejor práctica cuando se cargan pesos.

### Parámetro weights_only:

- **`weights_only=True`**: (Recomendado) Solo carga tensores, evita código malicioso
- **`weights_only=False`**: Puede cargar objetos arbitrarios de Python, menos seguro

In [ ]:
# Crear una instancia del modelo sin pesos preentrenados (sin especificar ``weights``)
model = models.vgg16()

# Cargar los pesos guardados
model.load_state_dict(torch.load('model_weights.pth', weights_only=True))

# Configurar el modelo en modo evaluación
model.eval()

## Modo Evaluación: model.eval()

**Importante**: Asegúrate de llamar al método `model.eval()` antes de hacer inferencias para configurar las capas de dropout y batch normalization en modo evaluación. No hacer esto producirá resultados de inferencia inconsistentes.

### Diferencias entre model.train() y model.eval():

| Aspecto | `model.train()` | `model.eval()` |
|---------|----------------|----------------|
| **Dropout** | Activo (desactiva neuronas aleatoriamente) | Inactivo (todas las neuronas activas) |
| **Batch Normalization** | Usa estadísticas del batch actual | Usa estadísticas guardadas del entrenamiento |
| **Gradientes** | Se calculan por defecto | Requiere `torch.no_grad()` para desactivar |
| **Uso** | Durante entrenamiento | Durante evaluación/inferencia |

### Ejemplo de predicción correcta:

```python
model.eval()  # Configurar en modo evaluación
with torch.no_grad():  # Desactivar cálculo de gradientes
    predictions = model(input_data)
```

## Guardar y Cargar Modelos con Estructura

Al cargar pesos del modelo, necesitábamos instanciar primero la clase del modelo, porque la clase define la estructura de la red. Podríamos querer guardar la estructura de esta clase junto con el modelo, en cuyo caso podemos pasar `model` (y no `model.state_dict()`) a la función de guardado.

### Comparación de métodos:

| Método | Qué guarda | Ventajas | Desventajas |
|--------|------------|----------|-------------|
| `torch.save(model.state_dict(), ...)` | Solo pesos | Flexible, portable, seguro | Requiere definir la arquitectura |
| `torch.save(model, ...)` | Modelo completo | No necesita redefinir arquitectura | Menos portable, puede tener problemas de versión |

### ⚠️ Recomendación:

Como se describe en la documentación oficial de PyTorch, guardar el `state_dict` se considera la mejor práctica. Sin embargo, a continuación usamos el método de guardar el modelo completo como caso de uso alternativo.

In [ ]:
# Guardar el modelo completo (arquitectura + pesos)
torch.save(model, 'model.pth')

## Cargar Modelo Completo

Podemos cargar el modelo completo como se demuestra a continuación.

**Nota**: Usamos `weights_only=False` porque esto implica cargar el modelo completo, que es un caso de uso heredado para `torch.save`. Para nuevos proyectos, se recomienda usar `state_dict`.

In [ ]:
# Cargar el modelo completo
model = torch.load('model.pth', weights_only=False)

# Configurar en modo evaluación
model.eval()

## Guardar Checkpoints durante el Entrenamiento

Durante el entrenamiento, es común guardar checkpoints que incluyen no solo los pesos del modelo, sino también el estado del optimizador, la época actual, y la pérdida. Esto permite reanudar el entrenamiento exactamente donde se dejó.

### Guardar un checkpoint completo:

```python
# Durante el entrenamiento
checkpoint = {
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss,
}
torch.save(checkpoint, 'checkpoint.pth')
```

### Cargar un checkpoint:

```python
# Reanudar entrenamiento
checkpoint = torch.load('checkpoint.pth', weights_only=True)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
epoch = checkpoint['epoch']
loss = checkpoint['loss']

model.train()  # Configurar en modo entrenamiento
```

## Guardar y Cargar entre CPU y GPU

Al trabajar con modelos en diferentes dispositivos, necesitas considerar cómo se guardan y cargan los tensores.

### Guardar en GPU, cargar en CPU:

```python
# Guardar (el modelo está en GPU)
torch.save(model.state_dict(), 'model_weights.pth')

# Cargar en CPU
device = torch.device('cpu')
model = NeuralNetwork()
model.load_state_dict(torch.load('model_weights.pth', map_location=device, weights_only=True))
```

### Guardar en GPU, cargar en otra GPU:

```python
# Guardar
torch.save(model.state_dict(), 'model_weights.pth')

# Cargar en GPU 0
device = torch.device('cuda:0')
model = NeuralNetwork()
model.load_state_dict(torch.load('model_weights.pth', map_location='cuda:0', weights_only=True))
```

### Guardar en CPU, cargar en GPU:

```python
# Guardar
torch.save(model.state_dict(), 'model_weights.pth')

# Cargar en GPU
device = torch.device('cuda')
model = NeuralNetwork()
model.load_state_dict(torch.load('model_weights.pth', map_location=device, weights_only=True))
model.to(device)  # Mover el modelo a GPU
```

### Parámetro map_location:

- **`map_location='cpu'`**: Carga todos los tensores en CPU
- **`map_location='cuda:0'`**: Carga todos los tensores en GPU 0
- **`map_location=device`**: Carga en el dispositivo especificado

## Mejores Prácticas

### 1. Convención de Nombres
- Usa extensión `.pth` o `.pt` para archivos de PyTorch
- Incluye información relevante: `model_epoch10_acc95.pth`
- Para checkpoints: `checkpoint_epoch_{epoch}.pth`

### 2. Guardar Regularmente
```python
# Guardar cada N épocas
if (epoch + 1) % 5 == 0:
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }, f'checkpoint_epoch_{epoch+1}.pth')
```

### 3. Guardar el Mejor Modelo
```python
best_loss = float('inf')

for epoch in range(num_epochs):
    # ... entrenamiento ...
    
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pth')
```

### 4. Incluir Metadatos
```python
# Guardar con información adicional
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': epoch,
    'loss': loss,
    'accuracy': accuracy,
    'hyperparameters': {
        'learning_rate': lr,
        'batch_size': batch_size,
    },
}, 'model_with_metadata.pth')
```

### 5. Verificar Archivos Guardados
```python
# Verificar qué contiene el archivo guardado
checkpoint = torch.load('checkpoint.pth', weights_only=False)
print(checkpoint.keys())  # Ver qué claves están guardadas
```

## Errores Comunes y Soluciones

### 1. RuntimeError: Error(s) in loading state_dict
**Causa**: La arquitectura del modelo no coincide con los pesos guardados.

**Solución**: Asegúrate de que la definición del modelo sea exactamente igual a cuando se guardó.

### 2. Resultados inconsistentes después de cargar
**Causa**: Olvidaste llamar `model.eval()`.

**Solución**: Siempre llama `model.eval()` antes de hacer inferencias.

### 3. Problemas de memoria al cargar en GPU
**Causa**: El modelo es demasiado grande para la GPU.

**Solución**: Carga en CPU primero con `map_location='cpu'` o usa una GPU más grande.

### 4. FileNotFoundError al cargar
**Causa**: La ruta del archivo es incorrecta.

**Solución**: Usa rutas absolutas o verifica el directorio de trabajo actual.

```python
import os
print(os.getcwd())  # Ver directorio actual
```

## Resumen

En este notebook aprendimos:

1. **state_dict**: El diccionario que contiene todos los parámetros aprendidos del modelo
2. **Guardar pesos**: `torch.save(model.state_dict(), 'path')` - Método recomendado
3. **Cargar pesos**: `model.load_state_dict(torch.load('path', weights_only=True))`
4. **model.eval()**: Crucial para hacer inferencias correctas
5. **Checkpoints**: Guardar estado completo (modelo + optimizador + época)
6. **GPU/CPU**: Usar `map_location` para controlar dónde se cargan los tensores
7. **Mejores prácticas**: Guardar regularmente, incluir metadatos, usar nombres descriptivos

### Flujo de trabajo recomendado:

```python
# 1. Durante el entrenamiento
torch.save(model.state_dict(), 'model_weights.pth')

# 2. Para hacer predicciones
model = NeuralNetwork()
model.load_state_dict(torch.load('model_weights.pth', weights_only=True))
model.eval()

# 3. Hacer predicciones
with torch.no_grad():
    predictions = model(input_data)
```